In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [5]:
len(files)

72

In [7]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [6]:
# Q.2 Indexation and search
from minsearch import Index
def build_index(documents):
    index=Index(
        text_fields=['content'],
        keyword_fields=['filename']
    )
    index.fit(documents)
    return index

index=build_index(documents)

In [8]:
query = "How does the agentic loop keep calling the model until it stops?"

results = index.search(
    query=query,
    num_results=5
)

In [9]:
def build_context(search_results):
        lines = []

        for doc in search_results:
            lines.append(doc['filename'])
            lines.append('content: ' + doc['content'])
            #lines.append('A: ' + doc['answer'])
            lines.append('')

        return '\n'.join(lines).strip()
context=build_context(results)
print(context)

01-agentic-rag/lessons/14-agentic-loop.md
content: # The Agentic Loop

Video: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lesson, we did function calling by hand. We sent a
message and got back a function call. We ran it, sent the result back,
and got the answer.

That works for one function call. It breaks down when the model wants
to search several times, or when the first search misses the answer.
We don't know in advance how many calls the model will want. So we
need a loop that keeps calling the model and running tools until it's
done. An agent is exactly that.

## Anatomy of an agent

With the LLM in the driver's seat, we have an agent. It's an AI
assistant whose goal is to help the user.

An agent has three parts:

- Instructions, the role and behavior we want. We pass this as the
  `developer` message. The better the instructions, the better the
  agent helps.
- Tools, the functions the agent can call

In [10]:
INSTRUCTIONS = """
You are a helpful teaching assistant for the LLM Zoomcamp.

Answer questions using only the provided lesson content.

If the answer cannot be found in the provided context,
reply with "I don't know."
"""

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()

def build_prompt(query, search_results):
        context = build_context(search_results)
        return PROMPT_TEMPLATE.format(
            question=query, context=context
        )

prompt=build_prompt(query,results)
print(prompt)

QUESTION: How does the agentic loop keep calling the model until it stops?

CONTEXT:
01-agentic-rag/lessons/14-agentic-loop.md
content: # The Agentic Loop

Video: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lesson, we did function calling by hand. We sent a
message and got back a function call. We ran it, sent the result back,
and got the answer.

That works for one function call. It breaks down when the model wants
to search several times, or when the first search misses the answer.
We don't know in advance how many calls the model will want. So we
need a loop that keeps calling the model and running tools until it's
done. An agent is exactly that.

## Anatomy of an agent

With the LLM in the driver's seat, we have an agent. It's an AI
assistant whose goal is to help the user.

An agent has three parts:

- Instructions, the role and behavior we want. We pass this as the
  `developer` message. The better the 

In [11]:
from dotenv import load_dotenv
load_dotenv()

True

In [12]:
from openai import OpenAI
llm_client = OpenAI()

def llm(prompt,INSTRUCTIONS=INSTRUCTIONS,):
        input_messages = [
            {'role': 'developer', 'content': INSTRUCTIONS},
            {'role': 'user', 'content': prompt}
        ]

        response = llm_client.responses.create(
            model='gpt-5.4-mini',
            input=input_messages
        )

        return response.output_text
output=llm(prompt)

In [13]:
INSTRUCTIONS = """
You are a helpful teaching assistant for the LLM Zoomcamp.

Answer questions using only the provided lesson content.

If the answer cannot be found in the provided context,
reply with "I don't know."
"""

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()


class RAGBase:

    def __init__(
        self,
        index,
        llm_client,

        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        model='gpt-5.4-mini'
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.prompt_template = prompt_template
        self.model = model

    def search(self, query, num_results=5):

        return self.index.search(
            query,
            num_results=num_results,
        )

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(doc['filename'])
            lines.append('content: ' + doc['content'])
           
            lines.append('')

        return '\n'.join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer = self.llm(prompt)
        return answer

In [14]:
query='How does the agentic loop keep calling the model until it stops?'
assistant = RAGBase(
    index,
    llm_client=llm_client,
)

answer = assistant.rag(query)
print(answer)

Response(id='resp_056fc8b0dbcf2f18006a3507949c5c81a38033119d23f9ce79', created_at=1781860244.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_056fc8b0dbcf2f18006a35079563d081a392a96883d54bfdeb', content=[ResponseOutputText(annotations=[], text='The loop keeps calling the model by checking whether the response contains any `function_call` items.\n\n- It sends the current `messages` to the model.\n- If the model returns a function call, the code runs the tool, appends the tool result to `messages`, and sets `has_function_calls = True`.\n- After processing the response, if `has_function_calls == False`, the loop breaks.\n- So it repeats until the model returns a response with no function calls, which means it’s done and has a final answer.\n\nIn short: **no function calls this turn means stop; otherwise, keep looping and send the updated history back to the model.**', type='out

In [15]:
answer.usage.input_tokens

7114

In [16]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [17]:
# index chunk
index_chunk=build_index(chunks)

In [18]:
query='How does the agentic loop keep calling the model until it stops?'
assistant = RAGBase(
    index_chunk,
    llm_client=llm_client,
)

answer = assistant.rag(query)
print(answer)

Response(id='resp_03527ce07cd8b2b5006a3507b45ad081918cb22cfd476eaa6a', created_at=1781860276.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_03527ce07cd8b2b5006a3507b4c9388191bd1dea60d40b2982', content=[ResponseOutputText(annotations=[], text='The loop keeps calling the model inside a `while True` loop. After each call:\n\n- it checks the model output for any `function_call` items\n- if there are function calls, it runs them, appends the results to the message history, and keeps looping\n- if there are no function calls in that turn, it breaks out of the loop\n\nSo the stop condition is: **no function calls this turn means the model is done**.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=178

In [19]:
answer.usage.input_tokens

2297

In [21]:
def search(query: str) -> str:
    """Search for information in the course lessons."""
    results = index_chunk.search(query, num_results=5)

    return "\n\n".join(
        doc["content"] for doc in results
    )